# FBA analysis 

In [15]:
import cobra
from scipy.io import loadmat

In [17]:
tumor_path   = "/Users/valentinreateguirangel/dissertation_project/70%_ambiguity_binary_iMat/tumorModel.mat"
#healthy_path = '/Users/valentinreateguirangel/dissertation_project/10%_ambiguity_binary_iMat/healthyModel.mat'

### Loading Tumor model 

In [20]:
# Load the MATLAB .mat model
from cobra.io import load_matlab_model

tumor_model = load_matlab_model(
    tumor_path,
    variable_name="tumorModel"   
)

In [22]:
tumor_model

Name,tumorModel
Memory address,3099f6870
Number of metabolites,829
Number of reactions,934
Number of genes,2887
Number of groups,60
Objective expression,0
Compartments,"Cytosol, Extracellular, Nucleus, Golgi apparatus, Lysosome, Mitochondria, Endoplasmic reticulum, Peroxisome"


### Loading Healthy model 

In [25]:
"""# Load the MATLAB .mat model
from cobra.io import load_matlab_model

healthy_model = load_matlab_model(
    healthy_path,
    variable_name="healthyModel"   
)"""

'# Load the MATLAB .mat model\nfrom cobra.io import load_matlab_model\n\nhealthy_model = load_matlab_model(\n    healthy_path,\n    variable_name="healthyModel"   \n)'

In [27]:
#healthy_model

### Cheking for biomass reaction ID

In [30]:
# 1. List every reaction that still has a non-zero objective coefficient
c_rxns = [r for r in tumor_model.reactions if r.objective_coefficient != 0]
print("Objective-flagged reactions:", c_rxns)
# If this prints an empty list, no objective is set.

# 2. Search reaction *names* as well as IDs
hit = [r.id for r in tumor_model.reactions if 'biomass' in r.name.lower()]
print("Reactions whose *name* contains 'biomass':", hit)

Objective-flagged reactions: []
Reactions whose *name* contains 'biomass': []


### Loading full model to extract biomass_human and push it back to the models 

In [33]:
from cobra.io import read_sbml_model

full_model = read_sbml_model("/Users/valentinreateguirangel/Downloads/Human-GEM-1.19.0/model/Human-GEM.xml")

In [35]:
# Biomass reaction = MAR13082
biomass_rxn = full_model.reactions.get_by_id("MAR13082")

In [39]:
# Adding back to tumor and healhty model 
tumor_model.add_reactions([biomass_rxn.copy()])
tumor_model.objective = "MAR13082"

#healthy_model.add_reactions([biomass_rxn.copy()])
#healthy_model.objective = "MAR13082"

### FBA

In [44]:
# Activating Nutrients bakc in Python 
nutrient_bounds = {
    "MAR09034": -10,    # Glucose
    "MAR09048": -1000,  # Oxygen
    "MAR09041": -1,     # Arginine
    "MAR09047": -1,     # Lysine
    "MAR09049": -1,     # Methionine
    "MAR09043": -1,     # Phenylalanine
    "MAR09045": -1,     # Tryptophan
    "MAR09050": -1,     # Histidine
    "MAR09042": -1,     # Isoleucine
    "MAR09044": -1,     # Leucine
    "MAR09040": -1,     # Threonine
    "MAR09046": -1      # Valine
}

for rxn_id, lb in nutrient_bounds.items():
    try:
        rxn = tumor_model.reactions.get_by_id(rxn_id)
        rxn.lower_bound = lb
        rxn.upper_bound = 1000
    except KeyError:
        print(f"  {rxn_id} not found in model — skipping.")

#for rxn_id, lb in nutrient_bounds.items():
   # try:
      #  rxn = healthy_model.reactions.get_by_id(rxn_id)
      #  rxn.lower_bound = lb
       # rxn.upper_bound = 1000
   # except KeyError:
       # print(f"⚠️  {rxn_id} not found in model — skipping.")

  MAR09049 not found in model — skipping.
  MAR09043 not found in model — skipping.
  MAR09045 not found in model — skipping.
  MAR09050 not found in model — skipping.
  MAR09042 not found in model — skipping.
  MAR09044 not found in model — skipping.
  MAR09040 not found in model — skipping.
  MAR09046 not found in model — skipping.


In [46]:
# Tumor
tumor_solution = tumor_model.optimize()
print("Biomass flux =", tumor_solution.objective_value)

Biomass flux = 0.0


In [48]:
"""# Healhty
healthy_solution = healthy_model.optimize()
print("Biomass flux =", healthy_solution.objective_value)

SyntaxError: incomplete input (815426865.py, line 1)

In [50]:
print(tumor_model.reactions.get_by_id("MAR13082"))

MAR13082: 45.0 MAM01371c + 0.0267 MAM01721n + 45.0 MAM02040c + 0.1124 MAM02847c + 0.4062 MAM03161c + 0.0012 MAM10012c + 5.3375 MAM10013c + 0.2212 MAM10014c + 0.4835 MAM10015c --> 45.0 MAM01285c + 45.0 MAM02039c + 45.0 MAM02751c + MAM03970c


In [52]:
print(tumor_model.reactions.get_by_id("MAR13082").reaction)

45.0 MAM01371c + 0.0267 MAM01721n + 45.0 MAM02040c + 0.1124 MAM02847c + 0.4062 MAM03161c + 0.0012 MAM10012c + 5.3375 MAM10013c + 0.2212 MAM10014c + 0.4835 MAM10015c --> 45.0 MAM01285c + 45.0 MAM02039c + 45.0 MAM02751c + MAM03970c


In [54]:
print(tumor_model.objective.expression)

1.0*MAR13082 - 1.0*MAR13082_reverse_11d67


In [56]:
biomass_rxn = tumor_model.reactions.get_by_id("MAR13082")
print("Lower bound:", biomass_rxn.lower_bound)
print("Upper bound:", biomass_rxn.upper_bound)

Lower bound: 0.0
Upper bound: 1000.0


In [58]:
for rxn in tumor_model.exchanges:
    if "glucose" in rxn.name.lower() or "oxygen" in rxn.name.lower():
        print(rxn.id, rxn.name, rxn.lower_bound, rxn.upper_bound)

MAR09034 Exchange of glucose -10 1000


In [60]:
for rxn in tumor_model.reactions:
    flux = tumor_solution.fluxes[rxn.id]
    if abs(flux) > 1e-6:
        print(f"{rxn.id}: {flux}")

In [62]:
tumor_solution

,fluxes,reduced_costs
MAR03905,0.0,0.000000e+00
MAR04365,0.0,0.000000e+00
MAR04368,0.0,7.441139e-19
MAR04371,0.0,1.488228e-18
MAR04372,0.0,5.064597e-18
...,...,...
MAR20114,0.0,0.000000e+00
MAR20115,0.0,4.144740e-18
MAR20130,0.0,4.144740e-18
MAR20169,0.0,0.000000e+00


In [64]:
nonzero_fluxes = tumor_solution.fluxes[tumor_solution.fluxes.abs() > 1e-6]
print(f"{len(nonzero_fluxes)} reactions carry flux.")

0 reactions carry flux.
